## Question 1: What does the dataset look like?

In [16]:
import pandas as pd
netflix_credits = pd.read_csv("credits.csv")
netflix_credits.index = netflix_credits.index +1
print(netflix_credits.head(),"\n",netflix_credits.tail(),"\ncredits_table", netflix_credits.shape)

   person_id       id             name                character   role
1       3748  tm84618   Robert De Niro            Travis Bickle  ACTOR
2      14658  tm84618     Jodie Foster            Iris Steensma  ACTOR
3       7064  tm84618    Albert Brooks                      Tom  ACTOR
4       3739  tm84618    Harvey Keitel  Matthew 'Sport' Higgins  ACTOR
5      48933  tm84618  Cybill Shepherd                    Betsy  ACTOR 
        person_id         id                name     character      role
77797     736339  tm1059008    Adelaida Buscato     María Paz     ACTOR
77798     399499  tm1059008  Luz Stella Luengas  Karen Bayona     ACTOR
77799     373198  tm1059008         Inés Prieto         Fanny     ACTOR
77800     378132  tm1059008        Isabel Gaona        Cacica     ACTOR
77801    1950416  tm1059008      Julian Gaviria           NaN  DIRECTOR 
credits_table (77801, 5)


**77801 credits of actors and directors on Netflix titles with 5 columns contains:**
1. person_id: The actor or director's id on JustWatch (unique identifier)
2. id: The title id on JustWatch
3. name: The actor or director's name
4. character: The charactor name
5. role: ACTOR or DIRECTOR

In [17]:
netflix_casts = pd.read_csv("titles.csv")
netflix_casts.index = netflix_casts.index +1
print(netflix_casts.head(),"\ncasts_table", netflix_casts.shape)

         id                                title   type  \
1  ts300399  Five Came Back: The Reference Films   SHOW   
2   tm84618                          Taxi Driver  MOVIE   
3  tm154986                          Deliverance  MOVIE   
4  tm127384      Monty Python and the Holy Grail  MOVIE   
5  tm120801                      The Dirty Dozen  MOVIE   

                                                                                                                                                                                                                                                                                                                                                                                                                       description  \
1                                                                                                                                                                                                                              

**5850 shows/movies titles on Netflix with 15 columns contains:**

1. id: The title ID on JustWatch.
2. title: The name of the title.
3. show_type: TV show or movie.
4. description: A brief description.
5. release_year: The release year.
6. age_certification: The age certification.
7. runtime: The length of the episode (SHOW) or movie.
8. genres: A list of genres.
9. production_countries: A list of countries that produced the title.
10. seasons: Number of seasons if it's a SHOW.
11. imdb_id: The title ID on IMDB.
12. idb_score: Score on IMDB.
13. imdb_votes: Votes on IMDB.
14. tmdb_popularity: Popularity on TMDB.
15. tmdb_score: Score on TMDB. 

**Question 2: What dataset is using for content-based recommender system?**

In [18]:
import pandas as pd
import numpy as np

# 0. Load & Prepare Data
netflix_casts = netflix_casts[['id', 'title', 'genres', 'description']]
netflix_credits = netflix_credits[['id', 'name', 'role']]

pd.set_option('display.max_colwidth', None)

# Aggregate cast
cast = (
    netflix_credits[netflix_credits['role'] == 'ACTOR']
    .groupby('id')['name']
    .apply(lambda x: " ".join(x))
    .reset_index()
)

# Aggregate directors
directors = (
    netflix_credits[netflix_credits['role'] == 'DIRECTOR']
    .groupby('id')['name']
    .apply(lambda x: " ".join(x))
    .reset_index()
)

# Merge
df = netflix_casts.merge(cast, on='id', how='left')
df = df.merge(directors, on='id', how='left', suffixes=('_cast', '_director'))
df.rename(columns={'name_cast': 'cast', 'name_director': 'director'}, inplace=True)
df = df.fillna('')

# Remove duplicate titles
df = df.drop_duplicates(subset='title', keep='first').reset_index(drop=True)

# 0.1 Cleaning helpers
def clean_genre_string(g):
    g = str(g).lower()
    g = g.replace("[", "").replace("]", "")
    g = g.replace("'", "").replace('"', "")
    g = g.replace(",", " ")
    g = " ".join(g.split())
    return g

df['genres'] = df['genres'].apply(clean_genre_string)

# Cast & director already strings
df['cast'] = df['cast'].astype(str)
df['director'] = df['director'].astype(str)

# Build content field
df['content'] = (
    df['genres'] + " " +
    df['description'] + " " +
    df['cast'] + " " +
    df['director']
)

# 1. TF-IDF (row-only similarity)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['content'])

title_to_idx = pd.Series(df.index, index=df['title'])
idx_to_id = df['id'].to_dict()

# 2. Basic Recommender
def recommend(title, n=5):
    idx = title_to_idx[title]
    sim_scores = linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()
    top_idx = np.argsort(sim_scores)[::-1][1:n+1]
    return df.loc[top_idx, ['title', 'genres']]

# 3. Reason Generator (with cast member names)
def generate_reason(input_title, rec_row):
    input_row = df[df['title'] == input_title].iloc[0]
    reasons = []

    # GENRE OVERLAP
    g1 = set(input_row['genres'].split())
    g2 = set(rec_row['genres'].split())
    genre_overlap = g1.intersection(g2)
    if genre_overlap:
        reasons.append("similar genres: " + ", ".join(sorted(genre_overlap)))

    # CAST OVERLAP (full names)
    cast1 = set(input_row['cast'].split())
    cast2 = set(rec_row['cast'].split())
    cast_overlap = cast1.intersection(cast2)

    # Convert token overlap into readable names
    # Your cast field is stored as: "Robert De Niro Al Pacino"
    # So we treat each token as a name part
    if cast_overlap:
        readable = ", ".join(sorted(cast_overlap))
        reasons.append("shares cast members: " + readable)

    # DIRECTOR MATCH
    if input_row['director'] and input_row['director'] == rec_row['director']:
        reasons.append(f"same director ({input_row['director']})")

    # FALLBACK
    if not reasons:
        reasons.append("similar description and thematic content")

    return "; ".join(reasons)

# 4. Recommender With Reasons + IDs
def recommend_with_reasons(title, n=5):
    idx = title_to_idx[title]
    sim_scores = linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()
    top_idx = np.argsort(sim_scores)[::-1][1:n+1]

    results = []
    input_id = idx_to_id[idx]

    for i in top_idx:
        rec_row = df.loc[i]
        reason = generate_reason(title, rec_row)

        results.append({
            "input_id": input_id,
            "input_title": title,
            "recommended_id": rec_row['id'],
            "recommended_title": rec_row['title'],
            "genres": rec_row['genres'],
            "similarity_score": round(float(sim_scores[i]), 4),
            "reason": reason
        })

    return pd.DataFrame(results)

# 5. Test
recommend_with_reasons("The Irishman", n=5)

# 6. Export Full Recommendation Table
all_recs = []
for title in df['title'].unique():
    recs = recommend_with_reasons(title, n=5)
    all_recs.append(recs)

final_recs = pd.concat(all_recs, ignore_index=True)
final_recs.to_csv("netflix_recommendations.csv", index=False)
